In [30]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Installing libraries

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
!pip install -q requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [5]:
!pip uninstall -y langchain-community
!pip install -q "langchain-community<0.4"
!pip install -q requests==2.32.4

Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.31 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [49]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


Step 1a: INDEXING (DOCUMENT INGESTION)

In [11]:
video_id = "WUvTyaaNkzM" #only id not full url
try:
  yt_api = YouTubeTranscriptApi()
  transcript_list = yt_api.fetch(video_id, languages = ['en'])
  #flattening it into plain text
  transcript = ' '.join(chunk.text for chunk in transcript_list) #joins all the sentences of text into one big string
  print(transcript)
except TranscriptsDisabled:
  print("Transcript is disabled for this video")


Hey everyone, Grant here. This is the first video in a series on the essence of calculus, and I'll be publishing the following videos once per day for the next 10 days. The goal here, as the name suggests, is to really get the heart of the subject out in one binge-watchable set. But with a topic that's as broad as calculus, there's a lot of things that can mean, so here's what I have in mind specifically. Calculus has a lot of rules and formulas which are often presented as things to be memorized. Lots of derivative formulas, the product rule, the chain rule, implicit differentiation, the fact that integrals and derivatives are opposite, Taylor series, just a lot of things like that. And my goal is for you to come away feeling like you could have invented calculus yourself. That is, cover all those core ideas, but in a way that makes clear where they actually come from, and what they really mean, using an all-around visual approach. Inventing math is no joke, and there is a difference 

Step 1b- INDEXING (TEXT SPLITTING)

In [39]:
from requests.models import CONTENT_CHUNK_SIZE
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    separators = [" ", ",", "\n"],
    chunk_overlap = 40
)
chunks = text_splitter.create_documents([transcript])
print(len(chunks))

61


Step 1c and 1d: Indexing (Embedding Generation and storing in vector store)

In [40]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(
    api_key = api_key,
    model = 'text-embedding-3-small'
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [41]:
vector_store.index_to_docstore_id

{0: '2fe576f5-18cb-4606-a6cd-ff5dccb54cd8',
 1: '71a66951-a22f-4cc1-ac0a-2043137dbef8',
 2: '16fffd04-d5a9-4ddd-9729-6e55b2d01d75',
 3: '56f4229a-311f-40a2-8897-f927f2fcd447',
 4: '47efb3e6-f1f4-403e-ac00-b6c99ae1dafe',
 5: '71e3f564-2c73-4cd8-bc04-63a6a6873c72',
 6: '9274254b-fcff-4816-b271-fbdf9f779969',
 7: '274d8656-9b57-4f00-90f8-6636e1329ce4',
 8: '474dcf45-e2e0-4e88-826e-28535e570a6b',
 9: '923981fb-8056-4ae7-8a01-7de96dadb69d',
 10: '13894d3d-dedf-4b0a-beba-f621edd85068',
 11: '18f6f886-ee9f-4246-ad27-1c8cd90e4636',
 12: 'ecd1eeaa-af5b-45dc-b40d-790aea314ad2',
 13: 'd969905d-96b3-4740-bef4-7918e6fcc01c',
 14: 'e7e28bc6-2506-449b-b6d3-7b72f54ac885',
 15: '79194fe1-eeb8-427d-afea-deaf08dbd64a',
 16: '74e904db-f89a-4438-b47e-1d6dddb0c198',
 17: 'bf1f1112-116f-46f9-bc23-85216ed7930e',
 18: '18413eaf-290b-45b5-a60b-ba455b3d93ef',
 19: '4ca1c985-15f9-4964-9b38-0d80e4fe8b26',
 20: 'ec345bcd-40e4-4e69-b63e-29f5e6868b90',
 21: 'e1158c0b-de6c-4ce4-a76b-00fd2294622a',
 22: 'a4775168-ba28-

In [44]:
vector_store.get_by_ids(['50034795-897a-402c-9fa5-1a436b0efb35'])

[]

Step 2: Retrieval

In [45]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ca6447ea850>, search_kwargs={'k': 4})

In [46]:
retriever.invoke('where is calculus used')

[Document(id='9013d70b-2df7-4a68-b397-3414d28b011e', metadata={}, page_content='emerge in calculus. And what follows in this series are the details, for derivatives and integrals and more. At all points, I want you to feel that you could have invented calculus yourself, that if you drew the right pictures and played with each idea in just the right way, these formulas and'),
 Document(id='84bf8146-c1d9-491b-b0d3-e1df3b8dd26a', metadata={}, page_content='graph itself, is called the fundamental theorem of calculus. It ties together the two big ideas of integrals and derivatives, and shows how each one is an inverse of the other. All of this is only a high-level view, just a peek at some of the core ideas that emerge in calculus. And what follows in'),
 Document(id='71a66951-a22f-4cc1-ac0a-2043137dbef8', metadata={}, page_content="binge-watchable set. But with a topic that's as broad as calculus, there's a lot of things that can mean, so here's what I have in mind specifically. Calculus h

Step 3: Augmentation

In [58]:
llm = ChatOpenAI(

    temperature = 0.2
)

In [52]:
prompt = PromptTemplate(
    template = '''
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.
    {context}
    Question: {question}''',
    input_variables = ['context', 'question']

)

In [53]:
question = 'is calculus discussed in this video? if yes, then what was discussed'
retrieved_docs = retriever.invoke(question)

In [55]:
#concatinating all 4 docs
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [56]:
final_prompt = prompt.invoke({'context': context_text, 'question': question})

Step 4: Generation

In [59]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, calculus is discussed in this video. The video discusses the core ideas of calculus, including derivative formulas, the product rule, integrals, derivatives, and the fundamental theorem of calculus.


Building a chain

In [60]:
from langchain_core.runnables import RunnableParallel,RunnableSequence,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [64]:
def format_docs(retrieved_docs):
  context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
  return context_text

In [66]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [67]:
parallel_chain.invoke('what is calculus')

{'context': "binge-watchable set. But with a topic that's as broad as calculus, there's a lot of things that can mean, so here's what I have in mind specifically. Calculus has a lot of rules and formulas which are often presented as things to be memorized. Lots of derivative formulas, the product rule, the\n\ngraph itself, is called the fundamental theorem of calculus. It ties together the two big ideas of integrals and derivatives, and shows how each one is an inverse of the other. All of this is only a high-level view, just a peek at some of the core ideas that emerge in calculus. And what follows in\n\nemerge in calculus. And what follows in this series are the details, for derivatives and integrals and more. At all points, I want you to feel that you could have invented calculus yourself, that if you drew the right pictures and played with each idea in just the right way, these formulas and\n\nIn this initial video, I want to show how you might stumble into the core ideas of calcul

In [68]:
parser = StrOutputParser()

In [69]:
main_chain = parallel_chain | prompt | llm | parser

In [72]:
main_chain.invoke('what is told about calculus in this video')

'In this video, it is mentioned that calculus has a lot of rules and formulas that are often presented as things to be memorized, such as derivative formulas, the product rule, and the fundamental theorem of calculus. It also mentions that calculus ties together the two big ideas of integrals and derivatives.'